# EfficientNetB0 Re-run (fixed preprocessing) — Run on Colab (experiment E3 fix)

Final Year Project — Muhammad Asad Bukhari. **Runtime: GPU required**
(Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU).
Run the cells top to bottom: **Runtime -> Run all**. About 20–30 minutes.

**Why this re-run exists:** in the first E3 comparison (8 Nov 2026),
EfficientNetB0 scored 0.0273 — random chance for 38 classes — while
MobileNetV2 reached 0.8884 on the same pipeline. The cause was the
shared [0, 1] preprocessing: a frozen ImageNet base expects its own
`preprocess_input` ([0, 255] -> [-1, 1]). `src/compare_models.py` now
applies each base's own `preprocess_input`, adds a `--models` selector
so one model can be re-run, and prints a pooled-feature diagnostic
before training — if the base features are (near-)constant, the run
aborts that model instead of wasting 5 epochs.

**This notebook trains EfficientNetB0 only** (5 epochs, same seed and
learning rate as E3), and writes a separate file
`results/model_comparison_efficientnetb0_rerun.csv` — the original
`results/model_comparison.csv` is left untouched.


In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus, 'NO GPU — stop here. This re-run is not viable on CPU.'


In [ ]:
!git clone https://github.com/Asadshah-coder/leaf-disease-prediction.git
%cd leaf-disease-prediction
!pip install -q -r requirements.txt
!pip install -q gdown
!ls


In [ ]:
# Dataset: PlantVillage leaves (1.7 GB) from Google Drive — skip if already extracted.
import os
if not os.path.isdir('data/valid'):
    get_ipython().system('gdown --id 13_IbE48q2Nv3whIDxVf8OV60JzZGVySx -O "Plant Disease FYP.zip"')
    get_ipython().system('unzip -q "Plant Disease FYP.zip"')
    get_ipython().system('mkdir -p data && mv "Plant Disease FYP/train" data/train && mv "Plant Disease FYP/valid" data/valid')
get_ipython().system('echo "train images:"; find data/train -type f | wc -l')
get_ipython().system('echo "valid images:"; find data/valid -type f | wc -l')


In [ ]:
# Re-run EfficientNetB0 only, with its own preprocess_input (about 20-30 min on GPU).
# Do NOT edit the script; if the feature diagnostic aborts the model,
# report its printed feature stats instead of a result.
!python src/compare_models.py --data-dir data --epochs 5 --batch-size 32 --results-dir results --models efficientnetb0 --output model_comparison_efficientnetb0_rerun.csv


In [ ]:
print(open('results/model_comparison_efficientnetb0_rerun.csv').read())
!ls -lh results/model_comparison_efficientnetb0_rerun.csv


## Done — next steps

1. **Download `results/model_comparison_efficientnetb0_rerun.csv`**
   from the Files panel.
2. **Copy the number** into the experiment log
   `docs/experiments/E3.md`: the corrected EfficientNetB0 validation
   accuracy (and the feature diagnostic's printed stats) — only quote
   numbers from this run. If the diagnostic aborted the model, the
   re-run has no result: report that, with the printed stats.
3. **Push** the notebook + updated experiment log to GitHub; **never**
   commit the dataset (it is in `.gitignore` — check before pushing).
